# Polling percentage and proportion of seats won

Compare Conservative, Labour and Liberal Democrat polling with their share of seats in the training data. The denominator includes all constituency rows for each election, including seats won by other parties; it is not necessarily the total size of Parliament. The dashed line is one ordinary least-squares best fit across all three parties and elections, describing the training data.


In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# Relative paths support the notebook directory and the project root.
train_path = Path('../TEST_TRAIN/train.csv')
if not train_path.is_file():
    train_path = Path('TEST_TRAIN/train.csv')
# Keep the exploratory analysis historical too: never aggregate 2019 outcomes.
train_data = pd.concat([
    chunk.loc[chunk['election'] <= 2017].copy()
    for chunk in pd.read_csv(train_path, chunksize=512)
], ignore_index=True)

parties = ['Conservative', 'Labour', 'LD']
party_colours = {'Conservative': 'blue', 'Labour': 'red', 'LD': 'gold'}
winner_map = {'con': 'Conservative', 'lab': 'Labour', 'lib': 'LD'}
polling_by_election = train_data.groupby('election')[parties].first()
# Include all parties in the denominator.
total_seats = train_data.groupby('election').size()
seats_by_election = pd.crosstab(
    train_data['election'], train_data['winner'].map(winner_map)
).reindex(index=total_seats.index, columns=parties, fill_value=0)
seat_proportions = seats_by_election.div(total_seats, axis=0)

polling_long = polling_by_election.reset_index().melt(
    id_vars='election', value_vars=parties,
    var_name='party', value_name='polling'
)
seats_long = seat_proportions.reset_index().melt(
    id_vars='election', value_vars=parties,
    var_name='party', value_name='seat_proportion'
)
party_performance = polling_long.merge(
    seats_long, on=['election', 'party'], validate='one_to_one'
)
party_performance


In [ ]:
sns.set_theme(style='whitegrid')
fig, ax = plt.subplots(figsize=(10, 6))
for party in parties:
    party_data = party_performance.loc[
        party_performance['party'] == party
    ].dropna(subset=['polling', 'seat_proportion'])
    ax.scatter(
        party_data['polling'], party_data['seat_proportion'],
        color=party_colours[party], label=party, s=55
    )
    for row in party_data.itertuples(index=False):
        ax.annotate(
            str(int(row.election)), (row.polling, row.seat_proportion),
            textcoords='offset points', xytext=(5, 5), fontsize=8
        )

# Fit one line using all party-election observations together.
sns.regplot(
    data=party_performance, x='polling', y='seat_proportion',
    scatter=False, ci=None, color='black', ax=ax,
    label='Overall best fit',
    line_kws={'linestyle': '--', 'linewidth': 2}
)
ax.xaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.set_ylim(0, 1)
ax.set_xlabel('Polling percentage')
ax.set_ylabel('Proportion of seats won')
ax.set_title('Polling and seat proportion by election\nDashed line: linear best fit across all parties')
ax.legend()
fig.tight_layout()
plt.show()

# Express the plotted best-fit line with X in percentage points (35 means 35%).
import numpy as np
fit_pairs = party_performance[['polling', 'seat_proportion']].dropna()
slope, intercept = np.polyfit(
    fit_pairs['polling'] * 100, fit_pairs['seat_proportion'], 1
)
print(f'Best-fit equation: Y = {slope:.6f} X {intercept:+.6f}')
print('X = polling percentage (e.g. 35 for 35%)')
print('Y = proportion of seats won (e.g. 0.40 for 40%)')


### Equation of the best-fit line

The preceding cell recalculates the descriptive pooled line using only elections through 2017. X is polling in percentage points; Y is the proportion of constituency rows won. The forecast below instead fits a separate seat-count regression for each party.

In [ ]:
import numpy as np

# Use the same pooled party-election observations as the graph.
# Remove incomplete pairs; scaling polling to percentages does not change r.
pairs = party_performance[['polling', 'seat_proportion']].dropna().to_numpy()
n_observations = len(pairs)
if n_observations < 3 or np.any(np.ptp(pairs, axis=0) == 0):
    raise ValueError('Correlation requires at least three pairs and variation in both variables.')

# 1. Pearson correlation for the original sample.
original_correlation = np.corrcoef(pairs[:, 0], pairs[:, 1])[0, 1]

# 2-4. Resample whole rows with replacement, preserving each (polling, seats) pair.
# Each bootstrap sample has the same number of observations as the original.
# A fixed seed makes the results reproducible.
rng = np.random.default_rng(42)
n_bootstrap = 10_000
bootstrap_correlations = np.empty(n_bootstrap)
for i in range(n_bootstrap):
    indices = rng.choice(n_observations, size=n_observations, replace=True)
    sample = pairs[indices]
    # Pearson r is undefined if a resample has a constant variable.
    if np.any(np.ptp(sample, axis=0) == 0):
        bootstrap_correlations[i] = np.nan
    else:
        bootstrap_correlations[i] = np.corrcoef(sample[:, 0], sample[:, 1])[0, 1]

# Do not silently discard undefined correlations when calculating percentiles.
if not np.isfinite(bootstrap_correlations).all():
    raise ValueError('Some bootstrap correlations are undefined; inspect bootstrap_correlations.')

# 5-6. Central percentile intervals from the stored bootstrap distribution.
ci_90 = np.percentile(bootstrap_correlations, [5, 95])
ci_95 = np.percentile(bootstrap_correlations, [2.5, 97.5])

# 7. Report the original correlation and both confidence intervals.
bootstrap_results = {
    'original_correlation': original_correlation,
    '90% confidence interval': tuple(ci_90),
    '95% confidence interval': tuple(ci_95),
}
print(f'Original Pearson correlation: {original_correlation:.4f}')
print(f'90% percentile confidence interval: ({ci_90[0]:.4f}, {ci_90[1]:.4f})')
print(f'95% percentile confidence interval: ({ci_95[0]:.4f}, {ci_95[1]:.4f})')

# This observation-level bootstrap treats party-election rows as independent;
# it does not account for dependence between parties within the same election.


A problem with the current model selection process is that there is no accounting for the implicit constraint on the result as shown by the above correlation. This is not an explicit constraint because the voting of each consitency is indeed independent, however the proportion of seats for each party has a strong correlation with polling and this is not reflected in our pipeline. Specifically, the non-parametric models do not extrapolate so when faced with test data outside the range of the training set, this does not greatly affect the prediction as perhaps it should. For example, for the random-forest model, a polling level for Lib-Dems at 90% would affect the model similarly to a polling level of 28% because 28% is the greatest polling reflected in the training set. 

Solutions to this problem could be, using only parametric models because they can better extraploate. If the training data follows our assumption for the functional structure of the relationship, this would be likely to be able to extrapolate more effectively. Dangers of this are that extrapolation can be poor, especially if we assume an incorrect functional relationship between the predictors and response. Also, limiting the pipeline to just parametric models, stops us from reaping the benefits of the flexibility of non-parametric models.

On the other hand, we could penalise model prediction which doesn't approximately follow the number of seats we should get after the fact.



## XGBoost trained on elections through 2015

Reuse Models/xgboost_model.py so features, preprocessing, target encoding and the five-fold accuracy grid search are identical. Only elections up to and including 2015 are used for tuning and fitting. The fitted pipeline is available as xgboost_model_2015.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Support execution from the project root or the notebook directory.
project_root = Path.cwd()
if not (project_root / 'Models' / 'xgboost_model.py').is_file():
    project_root = project_root.parent
if not (project_root / 'Models' / 'xgboost_model.py').is_file():
    raise FileNotFoundError('Run from the project root or the notebook directory.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from Models.xgboost_model import train_xgboost

# Load independently so this section can run on its own.
from Models.constrained_xgboost import load_inputs
xgboost_data, _ = load_inputs(project_root / 'TEST_TRAIN' / 'train.csv')
election_years = pd.to_numeric(xgboost_data['election'], errors='raise')
xgboost_training_2015 = xgboost_data.loc[election_years <= 2015].copy()
if xgboost_training_2015.dropna(subset=['winner']).empty:
    raise ValueError('No labelled training rows are available through 2015.')

xgboost_model_2015 = train_xgboost(xgboost_training_2015)

print('Training elections:', sorted(xgboost_training_2015['election'].unique().tolist()))
print('Labelled training rows:', xgboost_training_2015['winner'].notna().sum())
classifier = xgboost_model_2015.named_steps['classifier']
print(f'Selected parameters: n_estimators={classifier.n_estimators}, max_depth={classifier.max_depth}')


In [ ]:
from Models.xgboost_model import FEATURE_COLUMNS

# Predict 2017 using the model fitted only on elections through 2015.
election_2017 = xgboost_data.loc[
    pd.to_numeric(xgboost_data['election'], errors='raise') == 2017
].copy().reset_index(drop=True)
if election_2017.empty:
    raise ValueError('No 2017 constituency rows are available.')

features_2017 = election_2017[FEATURE_COLUMNS]
# Probability columns must follow the fitted model's class order.
party_probabilities_2017 = pd.DataFrame(
    xgboost_model_2015.predict_proba(features_2017),
    columns=[f'probability_{party}' for party in xgboost_model_2015.classes_],
    index=election_2017.index,
)

predictions_2017 = election_2017[
    ['constituency_id', 'constituency_name', 'country/region', 'election', 'winner']
].rename(columns={'winner': 'actual_winner'})
predictions_2017['predicted_winner'] = xgboost_model_2015.predict(features_2017)
predictions_2017 = pd.concat([predictions_2017, party_probabilities_2017], axis=1)

# Probabilities are between 0 and 1 and sum to approximately 1 per constituency.
# Party labels retain the model's groupings, including natSW and oth.
predictions_2017


## Constrained 2019 XGBoost forecast

This section runs independently. Both XGBoost tuning/refitting and polling-to-seat regressions use only the available elections **2001, 2005, 2010, 2015 and 2017**. The 2019 frame contains identifiers and predictors only; no 2019 outcomes are used or evaluated. Source CSVs are unchanged.

The classifier follows notebook 03: the same 14 predictors, pandas dummy encoding aligned to training columns, LabelEncoder, multi:softprob, mlogloss, random state 42, and five-fold accuracy grid search over 20/35/50/100 trees and depths 2/3/4/5. Unlike that notebook's four-class filter, Other is retained as required. Parallelism is at the grid-search level.

Existing Conservative/Labour/LD columns supply national pre-election polling. Historical seat counts come from the existing winner labels, with one observation per election in each party's separate OLS fit. The data covers Great Britain; Northern Ireland is excluded. Historical constituency counts vary with election boundaries.

For each new-election prediction, the 95% interval is y_hat +/- t(0.975, n-2) * sqrt[SSE/(n-2) * (1 + 1/n + (x_new - x_mean)^2/Sxx)]. The leading 1 includes new-election residual uncertainty; this is not a confidence interval for the conditional mean. These are individual party intervals under the usual linear-model assumptions, based on only five elections. Integer bounds use ceiling(lower), floor(upper), intersected with [0, number of 2019 constituencies].

[SciPy MILP](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.milp.html) minimises total negative log probability with binary variables, exactly one party per seat, and bounds for Labour, Conservative and Liberal Democrat. Other and natSW remain unconstrained nationally. Only objective probabilities are clipped at 1e-15; saved probabilities are unmodified. Infeasible constraints raise an error without relaxation.


In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / 'Models' / 'constrained_xgboost.py').is_file():
    project_root = project_root.parent
if not (project_root / 'Models' / 'constrained_xgboost.py').is_file():
    raise FileNotFoundError('Run from the project root or the notebook directory.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from Models.constrained_xgboost import run_forecast

forecast_2019 = run_forecast(project_root / 'TEST_TRAIN' / 'train.csv')
xgboost_model_2017 = forecast_2019['grid_search'].best_estimator_
probabilities_2019 = forecast_2019['probabilities']
predictions_2019 = forecast_2019['predictions']
polling_models = forecast_2019['polling_models']
polling_seat_ranges_2019 = forecast_2019['seat_ranges']
seat_totals_2019 = forecast_2019['seat_totals']

print('Training elections:', forecast_2019['training_elections'])
print('Best XGBoost parameters:', forecast_2019['grid_search'].best_params_)
print('Optimiser:', forecast_2019['optimisation'].message)
print('Constituencies:', len(predictions_2019))
print('Changed assignments:', (predictions_2019.unconstrained_winner != predictions_2019.constrained_winner).sum())


Training elections: [2001, 2005, 2010, 2015, 2017]
Best XGBoost parameters: {'max_depth': 2, 'n_estimators': 20}
Optimiser: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Constituencies: 632
Changed assignments: 0


In [2]:
# Probabilities with identifiers only, before winner assignment.
probabilities_2019


,election,constituency_id,constituency_name,country/region,Labour probability,Conservative probability,Liberal Democrat probability,Other probability,natSW probability
0,2019,W07000049,Aberavon,Wales,0.984522,0.009322,0.003288,0.001485,0.001383
1,2019,W07000058,Aberconwy,Wales,0.240532,0.749023,0.002059,0.001208,0.007178
2,2019,S14000001,Aberdeen North,Scotland,0.105281,0.097236,0.003374,0.002598,0.791512
3,2019,S14000002,Aberdeen South,Scotland,0.016753,0.965608,0.001658,0.000973,0.015009
4,2019,S14000003,Airdrie & Shotts,Scotland,0.166480,0.028036,0.003403,0.003784,0.798296
...,...,...,...,...,...,...,...,...,...
627,2019,E14001059,Wythenshawe & Sale East,North West,0.955684,0.040553,0.001861,0.001101,0.000802
628,2019,E14001060,Yeovil,South West,0.003606,0.979689,0.015565,0.000357,0.000784
629,2019,W07000041,Ynys Mon,Wales,0.941921,0.043313,0.002213,0.002460,0.010093
630,2019,E14001061,York Central,Yorkshire and The Humber,0.954358,0.040496,0.003042,0.001303,0.000801


In [3]:
# Unrounded prediction intervals and feasible integer bounds are both retained.
polling_seat_ranges_2019


,polling_2019,predicted_seats,prediction_lower_95,prediction_upper_95,lower_seats,upper_seats,historical_elections,residual_df
party,,,,,,,,
Labour,0.332375,273.692860,95.097938,452.287782,96,452,5,3
Conservative,0.431042,347.295778,46.550590,648.040966,47,632,5,3
Liberal Democrat,0.120125,24.463002,-14.738705,63.664710,0,63,5,3


In [4]:
seat_totals_2019


,unconstrained_seats,constrained_seats
party,,
Labour,261,261
Conservative,327,327
Liberal Democrat,5,5
Other,2,2
natSW,37,37


In [5]:
# Final dataframe: all five probabilities and both winner assignments.
predictions_2019


,election,constituency_id,constituency_name,country/region,Labour probability,Conservative probability,Liberal Democrat probability,Other probability,natSW probability,unconstrained_winner,constrained_winner
0,2019,W07000049,Aberavon,Wales,0.984522,0.009322,0.003288,0.001485,0.001383,Labour,Labour
1,2019,W07000058,Aberconwy,Wales,0.240532,0.749023,0.002059,0.001208,0.007178,Conservative,Conservative
2,2019,S14000001,Aberdeen North,Scotland,0.105281,0.097236,0.003374,0.002598,0.791512,natSW,natSW
3,2019,S14000002,Aberdeen South,Scotland,0.016753,0.965608,0.001658,0.000973,0.015009,Conservative,Conservative
4,2019,S14000003,Airdrie & Shotts,Scotland,0.166480,0.028036,0.003403,0.003784,0.798296,natSW,natSW
...,...,...,...,...,...,...,...,...,...,...,...
627,2019,E14001059,Wythenshawe & Sale East,North West,0.955684,0.040553,0.001861,0.001101,0.000802,Labour,Labour
628,2019,E14001060,Yeovil,South West,0.003606,0.979689,0.015565,0.000357,0.000784,Conservative,Conservative
629,2019,W07000041,Ynys Mon,Wales,0.941921,0.043313,0.002213,0.002460,0.010093,Labour,Labour
630,2019,E14001061,York Central,Yorkshire and The Humber,0.954358,0.040496,0.003042,0.001303,0.000801,Labour,Labour


The constrained assignment may equal the ordinary argmax assignment when all three original seat totals lie inside their prediction intervals. The bounds need not bind, and the optimiser does not force totals toward the regression point predictions. Actual 2019 results remain reserved for subsequent evaluation.

## 2024 polling-to-seat interval boundaries

These are the previously calculated estimates using the **2001?2017 historical relationships** and the project's **2024 pre-election polling**. Values below are recorded to one decimal place for readability; this section does not train any models or read actual 2024 results.

**Confidence intervals** describe uncertainty in the conditional mean seat total. **Prediction intervals** describe uncertainty for a single new election and are the appropriate bounds for the proposed constituency assignment constraints. Both are 95% intervals, separately for each party.

The first table preserves the raw regression intervals, including negative lower limits. The second converts them to feasible integer seat bounds for the project's 632 Great Britain constituencies: round lower limits up, upper limits down, and restrict to 0?632. These are reference values only; this section does not run or change any constituency assignments.


In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

# Recorded 2024 estimates, using historical elections through 2017 only.
# Raw intervals are rounded to one decimal place, matching the reported figures.
seat_intervals_2024 = pd.DataFrame([
    {
        'party': 'Labour',
        'polling_percent': 38.27,
        'estimated_seats': 325.6,
        'confidence_lower_95': 249.0,
        'confidence_upper_95': 402.1,
        'prediction_lower_95': 148.7,
        'prediction_upper_95': 502.4,
    },
    {
        'party': 'Conservative',
        'polling_percent': 20.91,
        'estimated_seats': 111.2,
        'confidence_lower_95': -242.5,
        'confidence_upper_95': 464.9,
        'prediction_lower_95': -303.0,
        'prediction_upper_95': 525.4,
    },
    {
        'party': 'Liberal Democrat',
        'polling_percent': 11.00,
        'estimated_seats': 21.6,
        'confidence_lower_95': 1.9,
        'confidence_upper_95': 41.2,
        'prediction_lower_95': -18.2,
        'prediction_upper_95': 61.3,
    },
]).set_index('party')

print('2024 seat estimates and raw 95% intervals')
display(seat_intervals_2024.round(2))


2024 seat estimates and raw 95% intervals


,polling_percent,estimated_seats,confidence_lower_95,confidence_upper_95,prediction_lower_95,prediction_upper_95
party,,,,,,
Labour,38.27,325.6,249.0,402.1,148.7,502.4
Conservative,20.91,111.2,-242.5,464.9,-303.0,525.4
Liberal Democrat,11.00,21.6,1.9,41.2,-18.2,61.3


In [2]:
# Feasible integer bounds for the 632 constituencies in the project.
number_of_seats_2024 = 632
seat_bounds_2024 = pd.DataFrame(index=seat_intervals_2024.index)
for interval_type in ['confidence', 'prediction']:
    seat_bounds_2024[f'{interval_type}_lower_seats'] = (
        np.ceil(seat_intervals_2024[f'{interval_type}_lower_95'])
        .clip(lower=0, upper=number_of_seats_2024).astype(int)
    )
    seat_bounds_2024[f'{interval_type}_upper_seats'] = (
        np.floor(seat_intervals_2024[f'{interval_type}_upper_95'])
        .clip(lower=0, upper=number_of_seats_2024).astype(int)
    )

# Use prediction bounds, rather than confidence bounds, for a new election.
polling_seat_bounds_2024 = seat_bounds_2024[
    ['prediction_lower_seats', 'prediction_upper_seats']
].rename(columns={
    'prediction_lower_seats': 'lower_seats',
    'prediction_upper_seats': 'upper_seats',
})

print('Feasible integer boundaries (95%)')
display(seat_bounds_2024)
print('Prediction bounds for a potential 2024 constrained assignment')
display(polling_seat_bounds_2024)


Feasible integer boundaries (95%)


,confidence_lower_seats,confidence_upper_seats,prediction_lower_seats,prediction_upper_seats
party,,,,
Labour,249,402,149,502
Conservative,0,464,0,525
Liberal Democrat,2,41,0,61


Prediction bounds for a potential 2024 constrained assignment


,lower_seats,upper_seats
party,,
Labour,149,502
Conservative,0,525
Liberal Democrat,0,61
